In [1]:
#### Testing the compiler

"""
Main symbolic representation test for CoNAN.

This script automatically tests all multiplication functions
defined in examples.py and prints the corresponding symbolic
representation matrices.
"""

import inspect
import examples

import compiler as compiler
from compiler import AlgebraCompiler
import fast_decomposer as fastdec



############################################################################
# Function dimensions
############################################################################

FUNCTION_DIMENSIONS = {

    "mul_NTRU": 5,
    "mul_NTRUPrime": 5,
    "mul_Falcon": 8,
    "mul_NTRUPlus": 8,

    "mul_ETRU": 8,
    "mul_DiTRU": 8,
    "mul_LWE_DCC": 8,
    "mul_DiTRUPlus": 8,

    "mul_NTRU3": 27,
    "mul_MNTRU": 15,

    "mul_QTRU": 8,
    "mul_SQTRU": 8,

    "mul_BQTRU": 16,

    "mul_DiTRUplus_strong": 8,

    "mul_LWE_DCC_semi_direct": 8,
}


############################################################################
# Collect all multiplication functions automatically
############################################################################

FUNCTIONS = [

    (name, obj)

    for name, obj in inspect.getmembers(examples)

    if inspect.isfunction(obj)
    and name.startswith("mul_")
]


############################################################################
# Main test loop
############################################################################

for funcname, _ in FUNCTIONS:

    print("=" * 80)
    print(f"{funcname}")
    print("=" * 80)

    if funcname not in FUNCTION_DIMENSIONS:

        print(f"[SKIPPED] Missing dimension for {funcname}")
        print()

        continue

    n = FUNCTION_DIMENSIONS[funcname]

    try:

        ####################################################################
        # Initialize compiler
        ####################################################################

        compilerobj = AlgebraCompiler(
            filename="examples.py",
            funcname=funcname,
            dimension=n
        )

        ####################################################################
        # Extract structure tensor
        ####################################################################

        compilerobj.extract_structure_tensor()

        ####################################################################
        # Print symbolic matrix
        ####################################################################

        H = compilerobj.return_symbolic_matrix(
            mul_type='LL',
            variable='h'
        )

        print(H)
        print()

    except Exception as e:

        print(f"[FAILED] {funcname}")
        print(e)
        print()

mul_BQTRU
Matrix([[h0, h1, h2, h3, h4, h5, h6, h7, h8, h9, h10, h11, h12, h13, h14, h15], [h1, h0, h3, h2, h5, h4, h7, h6, h9, h8, h11, h10, h13, h12, h15, h14], [h2, h3, h0, h1, h6, h7, h4, h5, h10, h11, h8, h9, h14, h15, h12, h13], [h3, h2, h1, h0, h7, h6, h5, h4, h11, h10, h9, h8, h15, h14, h13, h12], [h4, h5, h6, h7, h0, h1, h2, h3, h12, h13, h14, h15, h8, h9, h10, h11], [h5, h4, h7, h6, h1, h0, h3, h2, h13, h12, h15, h14, h9, h8, h11, h10], [h6, h7, h4, h5, h2, h3, h0, h1, h14, h15, h12, h13, h10, h11, h8, h9], [h7, h6, h5, h4, h3, h2, h1, h0, h15, h14, h13, h12, h11, h10, h9, h8], [h8, h9, h10, h11, -h12, -h13, -h14, -h15, h0, h1, h2, h3, -h4, -h5, -h6, -h7], [h9, h8, h11, h10, -h13, -h12, -h15, -h14, h1, h0, h3, h2, -h5, -h4, -h7, -h6], [h10, h11, h8, h9, -h14, -h15, -h12, -h13, h2, h3, h0, h1, -h6, -h7, -h4, -h5], [h11, h10, h9, h8, -h15, -h14, -h13, -h12, h3, h2, h1, h0, -h7, -h6, -h5, -h4], [-h12, -h13, -h14, -h15, h8, h9, h10, h11, -h4, -h5, -h6, -h7, h0, h1, h2, h3], [-h13,

In [2]:
compilerobj = AlgebraCompiler(
    filename="examples.py",
    funcname='mul_NTRUPrime',
    dimension=5
)


instance_compiled = compilerobj.compile()  ### (attention checking associativity takes time)It compiles and return the dimension, the function name, and the tensor

symbolic_lattice = compilerobj.return_symbolic_matrix('LL') ### return the lattice in the symbolic form
basis_returned   = compilerobj.return_lattice('LL') ### return the basis for LL

In [6]:
n = 8
q = 256
base_var = 2/3.
H = compiler.construct_symbolic_matrix(
    filename="examples.py",
    funcname="mul_NTRU",
    dimension=n,
    mul_type='LL',
    variable='h'
)

print("Original symbolic matrix:")
print(H)

Original symbolic matrix:
Matrix([[h0, h1, h2, h3, h4, h5, h6, h7], [h7, h0, h1, h2, h3, h4, h5, h6], [h6, h7, h0, h1, h2, h3, h4, h5], [h5, h6, h7, h0, h1, h2, h3, h4], [h4, h5, h6, h7, h0, h1, h2, h3], [h3, h4, h5, h6, h7, h0, h1, h2], [h2, h3, h4, h5, h6, h7, h0, h1], [h1, h2, h3, h4, h5, h6, h7, h0]])


In [7]:
decomposer = fastdec.SymbolicDecomposer(
    symbolic_matrix=H,
    n=n,
    q=q,
    base_var=base_var
)

In [8]:
decomposer.get_full_trees(verbose=True)

Checking:
Node (size (2, 2)):
⎡H₀  H₁⎤
⎢      ⎥
⎣H₁  H₀⎦
  ↓
    Node (size (1, 1)):
      Hom:
H₀ + H₁
  ↓
    Node (size (1, 1)):
      Hom:
H₀ - H₁
Checking:
Node (size (4, 4)):
⎡H₀  H₁  H₂  H₃⎤
⎢              ⎥
⎢H₃  H₀  H₁  H₂⎥
⎢              ⎥
⎢H₂  H₃  H₀  H₁⎥
⎢              ⎥
⎣H₁  H₂  H₃  H₀⎦
  ↓
    Node (size (1, 1)):
      Hom:
H₀ + H₁ + H₂ + H₃
  ↓
    Node (size (3, 3)):
⎡H₀ - H₁   H₁ - H₂   H₂ - H₃⎤
⎢                           ⎥
⎢-H₁ + H₃  H₀ - H₂   H₁ - H₃⎥
⎢                           ⎥
⎣-H₁ + H₂  -H₂ + H₃  H₀ - H₃⎦
      ↓
        Node (size (1, 1)):
          Hom:
H₀ - H₁ + H₂ - H₃
      ↓
        Node (size (2, 2)):
⎡H₀ - H₂   H₁ - H₃⎤
⎢                 ⎥
⎣-H₁ + H₃  H₀ - H₂⎦
          ↓
            Node (size (1, 1)):
              Hom:
H₀ - ⅈ⋅H₁ - H₂ + ⅈ⋅H₃
          ↓
            Node (size (1, 1)):
              Hom:
H₀ + ⅈ⋅H₁ - H₂ - ⅈ⋅H₃


[{'matrix': Matrix([
  [H0, H1],
  [H1, H0]]),
  'children': [{'matrix': Matrix([[H0 + H1]]),
    'children': [],
    'dim': 4,
    'root_dim': 8,
    'variance': 1.3333333333333333},
   {'matrix': Matrix([[H0 - H1]]),
    'children': [],
    'dim': 4,
    'root_dim': 8,
    'variance': 1.3333333333333333}],
  'dim': 8,
  'root_dim': 8,
  'variance': 0.6666666666666666,
  'transform': (Matrix([
   [1/2,  1/2],
   [1/2, -1/2]]),
   Matrix([
   [1,  1],
   [1, -1]]))},
 {'matrix': Matrix([
  [H0, H1, H2, H3],
  [H3, H0, H1, H2],
  [H2, H3, H0, H1],
  [H1, H2, H3, H0]]),
  'children': [{'matrix': Matrix([[H0 + H1 + H2 + H3]]),
    'children': [],
    'dim': 2,
    'root_dim': 8,
    'variance': 2.6666666666666665},
   {'matrix': Matrix([
    [ H0 - H1,  H1 - H2, H2 - H3],
    [-H1 + H3,  H0 - H2, H1 - H3],
    [-H1 + H2, -H2 + H3, H0 - H3]]),
    'children': [{'matrix': Matrix([[H0 - H1 + H2 - H3]]),
      'children': [],
      'dim': 2,
      'root_dim': 8,
      'variance': 2.6666666666